# Exp1: Final model evaluation

This notebook uses the shared evaluation engine to compare the final models: logistic regression, histogram gradient boosting, feature selection and demographic inclusion. The all-sources models use the same 31 historical features; age is retained throughout.

**Run All** to refresh the final comparison. Four models are trained from the saved feature table; the full gradient-boosting and demographic-variant predictions are loaded from the saved score files. Existing hyperparameters and the training-derived feature-selection list are reused. All intermediate predictions stay in memory.

With `SAVE = True`, the notebook replaces `Exp_1/exp1_outputs/exp1_metrics.csv` (six models) and `exp1_comparisons.csv` (five planned comparisons × two metrics). Earlier trials are not appended. Holm correction covers all ten planned tests.

For a quick comparison using only the two saved gradient-boosting variants, set `RUN_FINAL_COMPARISON = False`. Quick mode displays results without overwriting the complete final CSVs.

In [1]:
import importlib.util
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = next((parent for parent in (Path.cwd(), *Path.cwd().parents)
                if (parent / "Exp_1/exp1.ipynb").is_file()
                and (parent / "outputs/outputs/exp1/eval_exp1.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Run from the project root or a folder inside bt4012_group6.")
HERE = PROJECT / "Exp_1"
OUTPUT = HERE / "exp1_outputs"
ENGINE = PROJECT / "outputs/outputs/exp1/eval_exp1.py"
spec = importlib.util.spec_from_file_location("shared_exp1_evaluation", ENGINE)
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Evaluator:", ENGINE)
print("Predictions:", OUTPUT)

Evaluator: /Users/yuran/Desktop/BT Group Project/bt4012_group6/outputs/outputs/exp1/eval_exp1.py
Predictions: /Users/yuran/Desktop/BT Group Project/bt4012_group6/Exp_1/exp1_outputs


## 1. Your models

In [2]:
RUN_FINAL_COMPARISON = True
N_BOOT = 1000
SAVE = True

In [3]:
def build_final_models(project, include_full=True):
    """Final model-family, feature-selection and demographic comparisons."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    output = project / "Exp_1/exp1_outputs"
    demographic = pd.read_csv(output / "sensitivity_scores.csv")
    best = pd.read_csv(output / "best_model_scores.csv")
    with_attributes = demographic[["SK_ID_CURR", "score_with_attributes"]].rename(
        columns={"score_with_attributes": "score"})
    saved = {"GBM all sources": best, "GBM all sources + demographics": with_attributes}
    if not include_full:
        return saved

    metadata = json.loads((output / "model_metadata.json").read_text())
    specification = metadata["sensitivity"]
    all_columns = specification["feature_sets"]["Without gender and marital status"]
    history = {c for cols in metadata["history_features"].values() for c in cols}
    application = [c for c in all_columns if c not in history]
    selected = metadata["selected_features"]
    assert len(history) == 31 and len(application) == 120
    assert set(selected).issubset(all_columns)
    features = pd.read_parquet(output / "features.parquet")
    split = pd.read_csv(project / "outputs/outputs/clean_data/splits.csv")[["SK_ID_CURR", "TARGET", "split"]]
    features = features.merge(split, on="SK_ID_CURR", validate="one_to_one")
    train = features["split"].eq("train")
    scored = features["split"].isin(["val", "test"])
    # Reuse the existing LR regularisation settings; do not tune on the test set.
    models = {
        "LR application": (application, make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(C=1.0, max_iter=2000))),
        "LR all sources": (all_columns, make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(C=0.1, max_iter=2000))),
        "GBM application": (application, HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
        "GBM selected features": (selected, HistGradientBoostingClassifier(
                              **specification["parameters"], **specification["fixed_settings"])),
    }
    predictions = {}
    for label, (columns, model) in models.items():
        X = pd.get_dummies(features[columns], dummy_na=True).astype("float32")
        model.fit(X.loc[train], features.loc[train, "TARGET"])
        predictions[label] = features.loc[scored, ["SK_ID_CURR"]].assign(
            score=model.predict_proba(X.loc[scored])[:, 1])
        print(f"Trained {label}: {X.shape[1]} inputs; predictions kept in memory")
    return {
        "LR application": predictions["LR application"],
        "LR all sources": predictions["LR all sources"],
        "GBM application": predictions["GBM application"],
        "GBM all sources": best,
        "GBM selected features": predictions["GBM selected features"],
        "GBM all sources + demographics": with_attributes,
    }

def evaluate_final_models(engine, models, n_boot=1000):
    """Use the shared metric and paired-test routines for five planned comparisons."""
    rows, evaluated = [], {}
    test_ids = set(engine.load_splits().query("split == 'test'")["SK_ID_CURR"])
    val_ids = set(engine.load_splits().query("split == 'val'")["SK_ID_CURR"])
    for label, values in models.items():
        frame = engine.load_predictions(values)
        if frame["SK_ID_CURR"].duplicated().any() or not np.isfinite(frame["score"]).all():
            raise ValueError(f"{label}: invalid or duplicate scores")
        val = frame[frame["split"].eq("val")].sort_values("SK_ID_CURR")
        test = frame[frame["split"].eq("test")].sort_values("SK_ID_CURR")
        if set(test["SK_ID_CURR"]) != test_ids or set(val["SK_ID_CURR"]) != val_ids:
            raise ValueError(f"{label}: must score every shared validation and test applicant")
        result = engine.evaluate(test.TARGET, test.score, y_val=val.TARGET, s_val=val.score, n_boot=n_boot)
        row = {"model": label, "threshold": result.attrs["threshold"]}
        for metric, values in result.iterrows():
            row[metric] = values["value"]
            row[metric + "_ci_low"] = values["ci_low"]
            row[metric + "_ci_high"] = values["ci_high"]
        rows.append(row)
        evaluated[label] = {"val": val, "test": test, "threshold": result.attrs["threshold"]}
    pairs = [
        ("LR all sources", "LR application"),
        ("GBM all sources", "GBM application"),
        ("GBM all sources", "LR all sources"),
        ("GBM selected features", "GBM all sources"),
        ("GBM all sources + demographics", "GBM all sources"),
    ]
    comparisons = []
    for label, baseline in pairs:
        if label not in evaluated or baseline not in evaluated:
            continue
        trial, base = evaluated[label]["test"], evaluated[baseline]["test"]
        assert np.array_equal(trial.SK_ID_CURR, base.SK_ID_CURR)
        for metric in ("pr_auc", "roc_auc"):
            result = engine.compare_models(base.TARGET, trial.score, base.score, metric, n_boot)
            comparisons.append({"model": label, "vs_baseline": baseline, "metric": metric,
                "delta": result["delta"], "ci_low": result["ci_low"], "ci_high": result["ci_high"],
                "p_bootstrap": result["p_bootstrap"],
                "p_delong": result["delong_roc_auc"]["p_value"] if metric == "roc_auc" else np.nan})
    # Adjust across the complete set of planned tests, rather than separately by section.
    return pd.DataFrame(rows).set_index("model"), engine._add_p_values(pd.DataFrame(comparisons)), evaluated

MODELS = build_final_models(PROJECT, include_full=RUN_FINAL_COMPARISON)

Trained LR application: 250 inputs; predictions kept in memory
Trained LR all sources: 281 inputs; predictions kept in memory
Trained GBM application: 250 inputs; predictions kept in memory
Trained GBM selected features: 236 inputs; predictions kept in memory


## 2. Results

In [4]:
table, comps, preds = evaluate_final_models(ev, MODELS, n_boot=N_BOOT)

### Test-set metrics
One row per model. Each cell shows **value (95% confidence interval)**. The threshold used for precision, recall and
F1 was chosen on the **validation** set (the value that maximises F1), so the test set stays untouched.

| Metric | Role | Meaning |
|---|---|---|
| `pr_auc` | **primary** | precision-recall area; the most informative with only ~8% defaulters (8% = random) |
| `roc_auc` | secondary | how well the model ranks defaulters above non-defaulters (0.5 = random, 1 = perfect); unaffected by the default rate, and the standard credit-scoring metric |
| `precision` / `recall` / `f1` | operating point | at the chosen threshold: share of flagged who default / share of defaulters flagged / balance of both |

Compare models on `pr_auc` and `roc_auc` (the significance tests below). Precision, recall and F1 depend on where each
model's threshold lands, so read them as one example operating point, not as the model comparison. The max-F1 threshold also
treats a missed defaulter and a wrongly flagged applicant as equally costly; a lender would normally set the cut-off
from the cost of each error and its review capacity.

In [5]:
display(ev.pretty_metrics(table))

,threshold,pr_auc,roc_auc,precision,recall,f1
model,,,,,,
LR application,0.148,0.205 (0.186-0.225),0.733 (0.718-0.746),0.212 (0.196-0.230),0.376 (0.349-0.404),0.271 (0.252-0.291)
LR all sources,0.158,0.224 (0.204-0.247),0.746 (0.731-0.760),0.237 (0.220-0.257),0.390 (0.364-0.417),0.295 (0.276-0.315)
GBM application,0.166,0.215 (0.196-0.237),0.741 (0.726-0.755),0.235 (0.217-0.254),0.350 (0.324-0.376),0.281 (0.260-0.301)
GBM all sources,0.144,0.250 (0.228-0.273),0.758 (0.743-0.772),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321)
GBM selected features,0.146,0.249 (0.228-0.272),0.758 (0.743-0.772),0.233 (0.216-0.251),0.433 (0.406-0.461),0.303 (0.283-0.323)
GBM all sources + demographics,0.126,0.248 (0.226-0.271),0.759 (0.744-0.772),0.217 (0.203-0.232),0.500 (0.472-0.528),0.302 (0.285-0.320)


### Planned comparisons

1. LR all sources versus LR application: historical information in a linear model.
2. GBM all sources versus GBM application: historical information in a nonlinear model.
3. GBM all sources versus LR all sources: model family on the same inputs.
4. GBM selected features versus GBM all sources: feature selection.
5. GBM with demographics versus GBM all sources: gender and marital status.

Positive differences favour the first model. The verdict uses Holm-adjusted p-values over all ten tests. Confidence intervals are unadjusted; an interval excluding zero can still have a non-significant adjusted result.

In [6]:
display(ev.pretty_comparisons(comps))

,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,LR all sources,LR application,pr_auc,+0.0198 (+0.0101 to +0.0300),0,0,significantly better
1,LR all sources,LR application,roc_auc,+0.0131 (+0.0070 to +0.0192),2.5e-05,0.00015,significantly better
2,GBM all sources,GBM application,pr_auc,+0.0351 (+0.0233 to +0.0455),0,0,significantly better
3,GBM all sources,GBM application,roc_auc,+0.0165 (+0.0106 to +0.0225),1.7e-07,1.2e-06,significantly better
4,GBM all sources,LR all sources,pr_auc,+0.0258 (+0.0121 to +0.0380),0,0,significantly better
5,GBM all sources,LR all sources,roc_auc,+0.0119 (+0.0042 to +0.0194),0.0021,0.011,significantly better
6,GBM selected features,GBM all sources,pr_auc,-0.0012 (-0.0055 to +0.0025),0.58,1,no significant difference
7,GBM selected features,GBM all sources,roc_auc,+0.0001 (-0.0022 to +0.0026),0.94,1,no significant difference
8,GBM all sources + demographics,GBM all sources,pr_auc,-0.0023 (-0.0065 to +0.0026),0.47,1,no significant difference
9,GBM all sources + demographics,GBM all sources,roc_auc,+0.0012 (-0.0015 to +0.0041),0.38,1,no significant difference


## 3. Save the final comparison

Only the complete six-model comparison replaces the final CSV files. Quick mode is display-only.

In [7]:
if SAVE and RUN_FINAL_COMPARISON:
    table.to_csv(OUTPUT / "exp1_metrics.csv")
    comps.to_csv(OUTPUT / "exp1_comparisons.csv", index=False)
    print("Saved:", OUTPUT / "exp1_metrics.csv", "and", OUTPUT / "exp1_comparisons.csv")
else:
    print("Results displayed only; final CSVs unchanged.")

Saved: /Users/yuran/Desktop/BT Group Project/bt4012_group6/Exp_1/exp1_outputs/exp1_metrics.csv and /Users/yuran/Desktop/BT Group Project/bt4012_group6/Exp_1/exp1_outputs/exp1_comparisons.csv
